# Vectorless RAG (PageIndex) — Basics Walkthrough

A stripped-down version of `vectorless_rag_pipeline.ipynb`: **one document, one
question**, no batch loops, no retry wrapper, no cost tracker, no resumability.
Every stage of PageIndex is its own cell so you can run it line by line and see
exactly what goes in and comes out, before wrapping any of it in your own
functions.

Stages: build tree → navigate tree → fetch selected section text → generate answer.

---
## Setup — mount Drive, load the repo, load API key

In [1]:
import os, sys, json, re
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from google.colab import drive

drive.mount('/content/drive')

REPO_ROOT = Path('/content/drive/MyDrive/financebench_project')
if not (REPO_ROOT / "data" / "financebench_open_source.jsonl").exists():
    print(f"Repo not found at {REPO_ROOT} — cloning ...")
    !git clone https://github.com/shaliqsv/financebench-rag-thesis.git "{REPO_ROOT}"
else:
    print(f"Repo already present at {REPO_ROOT} — syncing to latest main ...")
    !git -C "{REPO_ROOT}" fetch origin
    !git -C "{REPO_ROOT}" checkout main
    !git -C "{REPO_ROOT}" pull origin main --no-rebase --no-edit

sys.path.insert(0, str(REPO_ROOT))

DATA_DIR = REPO_ROOT / "data"
PDF_DIR  = REPO_ROOT / "pdfs"

load_dotenv(REPO_ROOT / ".env", override=True)
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY", "")
print("GOOGLE_API_KEY:", "ok" if GOOGLE_API_KEY else "MISSING - fill in .env")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Repo already present at /content/drive/MyDrive/financebench_project — syncing to latest main ...
D	experiments/results/vector_rag_results.jsonl
Already on 'main'
Your branch is up to date with 'origin/main'.
From https://github.com/shaliqsv/financebench-rag-thesis
 * branch            main       -> FETCH_HEAD
Already up to date.
GOOGLE_API_KEY: ok


In [ ]:
# pinned, not "latest pageindex" -- so this notebook keeps behaving the same
# way weeks from now instead of picking up whatever VectifyAI ships next
%pip install -q python-dotenv pandas litellm PyPDF2 pyyaml pageindex==0.2.12 nest_asyncio

In [10]:
import litellm
import nest_asyncio
from pageindex import ConfigLoader, get_page_tokens, create_node_mapping, llm_completion, page_index_main

litellm.drop_params = True

# page_index_main runs its own asyncio.run() internally, but this notebook's
# kernel is already inside a running event loop -- nest_asyncio patches the
# loop so a nested asyncio.run() is allowed instead of raising RuntimeError.
nest_asyncio.apply()

# gemini-3.5-flash's free tier caps at 5 requests/minute, and tree-building
# fires many concurrent calls -- same wall vector_rag_pipeline.ipynb hit and
# fixed the same way. flash-lite has a much bigger free-tier allowance.
MODEL = "gemini/gemini-3.1-flash-lite"

PageIndex's settings (chunk size limits, whether to add node summaries, etc.)
all live in one config object. `ConfigLoader` starts from the library's
defaults and overrides just `model`.

In [11]:
opt = ConfigLoader().load({"model": MODEL})
opt

namespace(toc_check_page_num=20,
          max_page_num_each_node=10,
          max_token_num_each_node=20000,
          if_add_node_id='yes',
          if_add_node_summary='yes',
          if_add_doc_description='no',
          if_add_node_text='no',
          model='gemini/gemini-3.1-flash-lite',
          index_model='gemini/gemini-3.1-flash-lite',
          summary_model='gemini/gemini-3.1-flash-lite',
          chat_model='gemini/gemini-3.1-flash-lite',
          retrieve_model='gemini/gemini-3.1-flash-lite')

---
## Pick one question to work with

In [52]:
df_questions = pd.read_json(DATA_DIR / "financebench_open_source.jsonl", lines=True)
df_meta      = pd.read_json(DATA_DIR / "financebench_document_information.jsonl", lines=True)
df           = pd.merge(df_questions, df_meta, on=["doc_name", "company"])

row = df.iloc[0]
row['justification']

'The metric capital expenditures was directly extracted from the company 10K. The line item name, as seen in the 10K, was: Purchases of property, plant and equipment (PP&E).'

In [29]:
row = df.iloc[146]
row

,146
financebench_id,financebench_id_00566
company,Verizon
doc_name,VERIZON_2022_10K
question_type,domain-relevant
question_reasoning,Numerical reasoning
domain_question_num,dg22
question,Has Verizon increased its debt on balance shee...
answer,No. Verizon's debt decreased by $229 million.
justification,debt change = debt in 2022 - debt in 2021 = 15...
dataset_subset_label,OPEN_SOURCE


---
## Stage 1 — Build the PageIndex tree

Parses the PDF once into a hierarchical section tree (titles, node_ids,
LLM-written summaries). This is the "index" — no embeddings, no chunking.

In [54]:
pdf_path = PDF_DIR / f"{row.doc_name}.pdf"
tree_result = page_index_main(str(pdf_path), opt)
tree_result["structure"]

Parsing PDF...
start find_toc_pages
toc found
start detect_page_index
index found
process_toc_with_page_numbers
start_index: 1
start toc_transformer
start toc_index_extractor


ERROR:root:Error: litellm.InternalServerError: GeminiException InternalServerError - {
  "error": {
    "code": 500,
    "message": "Internal error encountered.",
    "status": "INTERNAL"
  }
}




Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

************* Retrying *************
Document validation: 160 pages, max allowed index: 160
start verify_toc
check all items


ERROR:root:Error: litellm.ServiceUnavailableError: GeminiException - {
  "error": {
    "code": 503,
    "message": "The service is currently unavailable.",
    "status": "UNAVAILABLE"
  }
}

ERROR:root:Error: litellm.ServiceUnavailableError: GeminiException - {
  "error": {
    "code": 503,
    "message": "The service is currently unavailable.",
    "status": "UNAVAILABLE"
  }
}

ERROR:root:Error: litellm.ServiceUnavailableError: GeminiException - {
  "error": {
    "code": 503,
    "message": "The service is currently unavailable.",
    "status": "UNAVAILABLE"
  }
}

ERROR:root:Error: litellm.ServiceUnavailableError: GeminiException - {
  "error": {
    "code": 503,
    "message": "The service is currently unavailable.",
    "status": "UNAVAILABLE"
  }
}

ERROR:root:Error: litellm.ServiceUnavailableError: GeminiException - {
  "error": {
    "code": 503,
    "message": "The service is currently unavailable.",
    "status": "UNAVAILABLE"
  }
}

ERROR:root:Error: litellm.InternalServer


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

************* Retrying *************

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

************* Retrying *************

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

************* Retrying *************

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

************* Retrying *************

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

************* Retrying *************

Give Feedback / Get Help: https://github.com/Berr

...

ERROR:root:Error: litellm.ServiceUnavailableError: GeminiException - {
  "error": {
    "code": 503,
    "message": "The service is currently unavailable.",
    "status": "UNAVAILABLE"
  }
}

ERROR:root:Error: litellm.ServiceUnavailableError: GeminiException - {
  "error": {
    "code": 503,
    "message": "The service is currently unavailable.",
    "status": "UNAVAILABLE"
  }
}




Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

************* Retrying *************

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

************* Retrying *************


ERROR:root:Error: litellm.ServiceUnavailableError: GeminiException - {
  "error": {
    "code": 503,
    "message": "The service is currently unavailable.",
    "status": "UNAVAILABLE"
  }
}

ERROR:root:Error: litellm.ServiceUnavailableError: GeminiException - {
  "error": {
    "code": 503,
    "message": "The service is currently unavailable.",
    "status": "UNAVAILABLE"
  }
}




Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

************* Retrying *************

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

************* Retrying *************


[{'title': 'Preface',
  'node_id': '0000',
  'start_index': 1,
  'end_index': 4,
  'summary': "This document is the 2018 Annual Report (Form 10-K) for 3M Company, filed with the U.S. Securities and Exchange Commission. It provides a comprehensive overview of the company's financial and operational status as of December 31, 2018. Key points include:\n\n*   **Corporate Information:** Basic identification details, stock exchange listings, and confirmation of 3M's status as a large accelerated filer.\n*   **Table of Contents:** A detailed outline of the report’s structure, covering business operations, risk factors, management’s discussion and analysis (MD&A), and comprehensive audited financial statements and notes.\n*   **Business Overview:** A summary of 3M’s identity as a diversified global technology company and a description of its five primary business segments: Industrial, Safety and Graphics, Health Care, Electronics and Energy, and Consumer.\n*   **Operational Context:** Informat

Separately, cache each page's raw text — the tree only stores which pages
each section spans (`start_index`/`end_index`), not the text itself. We'll
need this to pull out the actual words later.

In [55]:
page_list  = get_page_tokens(str(pdf_path), model=opt.model)
page_texts = [p[0] for p in page_list]
len(page_texts)

160

`create_node_mapping` flattens the nested tree into a flat `{node_id: node}`
dict, so we can look up any section by its id in one step instead of walking
the tree.

In [56]:
tree_result

{'doc_name': '3M_2018_10K.pdf',
 'structure': [{'title': 'Preface',
   'node_id': '0000',
   'start_index': 1,
   'end_index': 4,
   'summary': "This document is the 2018 Annual Report (Form 10-K) for 3M Company, filed with the U.S. Securities and Exchange Commission. It provides a comprehensive overview of the company's financial and operational status as of December 31, 2018. Key points include:\n\n*   **Corporate Information:** Basic identification details, stock exchange listings, and confirmation of 3M's status as a large accelerated filer.\n*   **Table of Contents:** A detailed outline of the report’s structure, covering business operations, risk factors, management’s discussion and analysis (MD&A), and comprehensive audited financial statements and notes.\n*   **Business Overview:** A summary of 3M’s identity as a diversified global technology company and a description of its five primary business segments: Industrial, Safety and Graphics, Health Care, Electronics and Energy, an

In [57]:
node_map = create_node_mapping(tree_result["structure"])
node_map

{'0000': {'title': 'Preface',
  'node_id': '0000',
  'start_index': 1,
  'end_index': 4,
  'summary': "This document is the 2018 Annual Report (Form 10-K) for 3M Company, filed with the U.S. Securities and Exchange Commission. It provides a comprehensive overview of the company's financial and operational status as of December 31, 2018. Key points include:\n\n*   **Corporate Information:** Basic identification details, stock exchange listings, and confirmation of 3M's status as a large accelerated filer.\n*   **Table of Contents:** A detailed outline of the report’s structure, covering business operations, risk factors, management’s discussion and analysis (MD&A), and comprehensive audited financial statements and notes.\n*   **Business Overview:** A summary of 3M’s identity as a diversified global technology company and a description of its five primary business segments: Industrial, Safety and Graphics, Health Care, Electronics and Energy, and Consumer.\n*   **Operational Context:** 

---
## Stage 2 — Navigate: ask the LLM which section(s) to look in

The model sees only titles/summaries (the tree), never the full section text
yet. It returns a ranked list of node_ids.

In [58]:
navigation_prompt = f"""You are navigating a financial filing's table-of-contents-style section tree \
to find the section(s) most likely to contain the answer to a question. You cannot see the section \
text yet, only titles and summaries. Return ONLY a JSON array of node_id strings, ordered from most \
to least likely to contain the answer (e.g. ["0003", "0007"]). Include at most 5 node_ids.

Question: {row.question}

Document tree:
{json.dumps(tree_result["structure"], indent=2)}"""

print(navigation_prompt[:1000])

You are navigating a financial filing's table-of-contents-style section tree to find the section(s) most likely to contain the answer to a question. You cannot see the section text yet, only titles and summaries. Return ONLY a JSON array of node_id strings, ordered from most to least likely to contain the answer (e.g. ["0003", "0007"]). Include at most 5 node_ids.

Question: What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement.

Document tree:
[
  {
    "title": "Preface",
    "node_id": "0000",
    "start_index": 1,
    "end_index": 4,
    "summary": "This document is the 2018 Annual Report (Form 10-K) for 3M Company, filed with the U.S. Securities and Exchange Commission. It provides a comprehensive overview of the company's financial and operational status as of December 31, 2018. Key points include:\n\n*   **Corporate Information:** Basic identification details, stock excha

In [59]:
navigation_response = llm_completion(MODEL, navigation_prompt)
navigation_response

'["0030", "0018", "0016"]'

Pull the JSON array out of the model's reply.

In [60]:
node_ids = json.loads(re.search(r"\[.*\]", navigation_response, re.DOTALL).group(0))
node_ids

['0030', '0018', '0016']

---
## Stage 3 — Fetch the raw text of just the selected section(s)

`start_index`/`end_index` are 1-indexed, inclusive physical page numbers —
that's why the slice below is `start - 1 : end`, not `start : end`.

In [61]:
selected_text = ""
for node_id in node_ids:
    node = node_map[node_id]
    start, end = node["start_index"], node["end_index"]
    section_text = "".join(page_texts[start - 1:end])
    selected_text += f"[Section: {node['title']}]\n{section_text}\n\n"

print(f"{len(selected_text)} characters")
print(selected_text[:500])

61500 characters
[Section: Consolidated Statement of Cash Flows for the years ended December 31, 2018, 2017 and 2016]
Table of Contents
 
3M Company and Subsidiaries
Consolidated Statement of Cash Flow
 
s
Years ended December 31
 
(Millions)
    
2018
    
2017
    
2016
 
Cash Flows from Operating Activities
 
 
 
 
 
 
 
 
 
 
Net income including noncontrolling interest
 
$
5,363
 
$
4,869
 
$
5,058
 
Adjustments to reconcile net income including noncontrolling interest to net cash
provided by operating acti


---
## Stage 4 — Generate the answer from only that text

This is the key difference from the long-context pipeline: the model never
sees the whole document, only the section(s) navigation picked.

In [62]:
generation_prompt = f"""You are a financial analyst. Answer the question using ONLY the filing \
sections provided below. Do not use outside knowledge or assume figures that aren't stated.

Work through it in this order:
1. Pull the exact figures or facts that bear on the question, quoting each with its label and \
period exactly as written (keep the units as stated, e.g. "$150,868 million", not a rounded or \
reworded version).
2. Answer the precise question asked. If it asks whether something increased/decreased or is \
higher/lower, decide the direction from the figures and state the size of the change. Watch the \
sign: a smaller current-year value than prior-year means a decrease.
3. If the sections don't contain what's needed, say the information isn't available instead of \
guessing.

Keep the final answer short and direct, in the style of these examples:
Q: Has the company increased its debt between 2021 and 2022?
FINAL ANSWER: No. Debt decreased by $229 million, from $150,868 million to $150,639 million.

Q: What was the company's operating margin in FY2023?
FINAL ANSWER: 14.2%.

Q: Did free cash flow grow year over year?
FINAL ANSWER: Yes. Free cash flow grew by $1.3 billion.

Question: {row.question}

Sections:
{selected_text}

Write your reasoning first, then on a new line write "FINAL ANSWER:" followed by the answer.
"""

model_output = llm_completion(MODEL, generation_prompt)
model_answer = (
    model_output.split("FINAL ANSWER:")[-1].strip()
    if "FINAL ANSWER:" in model_output
    else model_output.strip()
)

---
## Compare to the gold answer

In [63]:
print("question    :", row.question)
print("gold answer :", row.answer)
print("model answer:", model_answer)

question    : What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement.
gold answer : $1577.00
model answer: The FY2018 capital expenditure amount for 3M was $1,577 million.


In [64]:
row.justification

'The metric capital expenditures was directly extracted from the company 10K. The line item name, as seen in the 10K, was: Purchases of property, plant and equipment (PP&E).'